# SimCLR: Contrastive Learning of Visual Representations
> T. Chen, S. Kornblith, M. Norouzi, and G. Hinton, “A Simple Framework for Contrastive Learning of Visual Representations,” arXiv:2002.05709 [cs, stat], Jun. 2020, Available: https://arxiv.org/abs/2002.05709

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.transforms import v2 as T
from torchvision.datasets import ImageNet
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR


In [ ]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature 

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]

        # Normalize so dot product becomes cosine sim
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)
        z = torch.cat([z_i, z_j], dim=0)

        # Compute similarity matrix
        sim_matrix = torch.mm(z, z.t()) / self.temperature

        # Remove self-similarity from diagonal
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
        sim_matrix = sim_matrix.masked_fill(mask, float('-inf'))

        # Positive pairs
        labels = torch.arange(batch_size, device=z.device)
        labels = torch.cat([labels + batch_size, labels], dim=0)

        # Cross entropy
        loss = F.cross_entropy(sim_matrix, labels)
        return loss

class SimCLRModel(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights='DEFAULT')
        resnet.fc = nn.Identity()
        self.encoder = resnet
        self.projection = nn.Sequential(
            nn.Linear(2048 , 2048),  # Layer 1: Hidden Layer
            nn.ReLU(),                         # Non-linear Activation
            nn.Linear(2048, 128)  # Layer 2: Output Layer
        )
        self.transform = T.Compose([
            T.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomApply([
                T.ColorJitter(
                    brightness=0.8,
                    contrast=0.8,
                    saturation=0.8,
                    hue=0.2
                )
            ], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.RandomApply([
                T.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))
            ], p=0.5),
        ])

    def forward(self, x):
        x_i = self.transform(x)
        x_j = self.transform(x)
        z_i = self.projection(self.encoder(x_i))
        z_j = self.projection(self.encoder(x_j))
        return z_i, z_j
    
class LinearEvalModel(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(2048, num_classes)

        for param in self.encoder.parameters():
            param.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x)
        return self.classifier(h)

In [ ]:
def get_dataloaders(batch_sie=64):
    data_path = 'idk yet'
    val = ImageNet(root=data_path, split='val')
    train = ImageNet(root=data_path, split='train')
    # Set num_workers to number of CPU cors, pin_memory to True for faster GPU transfer
    train_loader = DataLoader(dataset=train, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(dataset=val, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
    return train_loader, val_loader

def apply_LR_scheduler(optimizer, total_epochs, warmup_epochs, steps_per_epoch):
    # Linear Warmup
    warmup_scheduler = LinearLR(optimizer, start_factor=0.0, end_factor=1.0, total_iters=warmup_epochs * steps_per_epoch)
    # Cosine Decay (without restarts)
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=(total_epochs - warmup_epochs) * steps_per_epoch, eta_min=0.0)
    # Combine LR schedulers sequentially
    combined_scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs * steps_per_epoch])

### Training Stage 1

In [ ]:
model = SimCLRModel()
criterion = NTXentLoss(temperature=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_loader, val_loader = get_dataloaders(batch_size=64)

steps_per_epoch = len(train_loader)
warmup_epochs = 10
total_epochs = 100

apply_LR_scheduler(optimizer, total_epochs, warmup_epochs, steps_per_epoch)

for epoch in range(total_epochs):
    for batch_idx, (images, targets) in enumerate(train_loader):
        #images = images.to(device)

        z_i, z_j = model(images)
        loss = criterion(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
encoder = model.encoder

### Training Stage 2

In [ ]:
linear_model = LinearEvalModel(encoder, 1000)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_loader, val_loader = get_dataloaders(batch_size=64)

steps_per_epoch = len(train_loader)
warmup_epochs = 10
total_epochs = 100

apply_LR_scheduler(optimizer, total_epochs, warmup_epochs, steps_per_epoch)

for epoch in range(total_epochs):
    for batch_idx, (images, targets) in enumerate(train_loader):
        #images, targets = images.to(device), targets.to(device)

        outputs = linear_model(images)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


### Evaluation